In [47]:
import geopandas as gp
import pandas as pd
import os
import numpy as np
import re
from collections import Counter

# Alabama 2024 General Election Results

In [48]:
path = './raw-from-source/2024-General Precinct Level Results/'

files = [i for i in os.listdir(path) if i != ".DS_Store"]

In [49]:
county_list = []
for file in files: 
    print(file)
    temp = pd.read_excel(path + file)
    county_name = file.split("-")[-1][0:-4]
    if county_name == "StClair":
        county_name = "St. Clair"
    temp["Party"] = temp["Party"].str.strip()
    temp["Party"] = temp["Party"].fillna("")
    temp["pivot_col"] = temp["Contest Title"].str.strip()+"-:-"+temp["Candidate"].str.strip()
    temp["pivot_col"] = np.where(temp["Party"]=="",temp["pivot_col"],temp["pivot_col"]+"-:-"+temp["Party"].str.strip())

    temp.drop(["Contest Title", "Party", "Candidate"], axis = 1, inplace = True)

    
    rename_dict = {i:i+"-:-"+county_name for i in temp.columns if i != "pivot_col"}
    temp.rename(columns = rename_dict, inplace = True)
    temp_transpose = temp.set_index("pivot_col").T
    temp_transpose.reset_index(inplace = True, drop = False)
    ser = temp_transpose["index"].value_counts()
    if ser[ser>1].shape[0]>0:
        raise ValueError
    temp_transpose["County"] = county_name
    #print(temp_transpose.head(10))
    #break
#     print(temp_transpose.head())
    county_list.append(temp_transpose)

2024-General-Marion.xls
2024-General-Greene.xls
2024-General-Jackson.xls
2024-General-Marengo.xls
2024-General-Escambia.xls
2024-General-Lauderdale.xls
2024-General-Calhoun.xls
2024-General-Monroe.xls
2024-General-Walker.xls
2024-General-DeKalb.xls
2024-General-Bibb.xls
2024-General-Cullman.xls
2024-General-Macon.xls
2024-General-Wilcox.xls
2024-General-Chambers.xls
2024-General-Randolph.xls
2024-General-Blount.xls
2024-General-Dale.xls
2024-General-Perry.xls
2024-General-Covington.xls
2024-General-Hale.xls
2024-General-Lowndes.xls
2024-General-Butler.xls
2024-General-Tallapoosa.xls
2024-General-Coosa.xls
2024-General-Pickens.xls
2024-General-Cherokee.xls
2024-General-Lee.xls
2024-General-Coffee.xls
2024-General-Cleburne.xls
2024-General-Crenshaw.xls
2024-General-Houston.xls
2024-General-Autauga.xls
2024-General-Sumter.xls
2024-General-Washington.xls
2024-General-Limestone.xls
2024-General-Clarke.xls
2024-General-Dallas.xls
2024-General-Lamar.xls
2024-General-Jefferson.xls
2024-General

In [50]:
comb = pd.concat(county_list)

In [51]:
comb = comb.fillna(0)

In [52]:
comb.head()

pivot_col,index,REGISTERED VOTERS - TOTAL-:-Registered Voters - Total,BALLOTS CAST - TOTAL-:-Ballots Cast - Total,BALLOTS CAST - BLANK-:-Ballots Cast - Blank,STRAIGHT PARTY-:-Alabama Democratic Party-:-DEM,STRAIGHT PARTY-:-Alabama Republican Party-:-REP,STRAIGHT PARTY-:-Over Votes,STRAIGHT PARTY-:-Under Votes,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES-:-Kamala D. Harris-:-DEM,PRESIDENT AND VICE PRESIDENT OF THE UNITED STATES-:-Donald J. Trump-:-REP,...,"SUPERINTENDENT, COLBERT COUNTY BOARD OF EDUCATION-:-Over Votes","SUPERINTENDENT, COLBERT COUNTY BOARD OF EDUCATION-:-Under Votes","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 2-:-David Yarber-:-REP","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 2-:-Write-In-:-NON","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 2-:-Over Votes","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 2-:-Under Votes","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 4-:-Thomas L Burgess-:-DEM","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 4-:-Write-In-:-NON","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 4-:-Over Votes","MEMBER, COLBERT COUNTY BOARD OF EDUCATION, DIST NO. 4-:-Under Votes"
0,ABSENTEE-:-Marion,0.0,451.0,1.0,36.0,324.0,0.0,91.0,55.0,389.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BACCUS UNION HALL-:-Marion,359.0,256.0,0.0,3.0,182.0,0.0,71.0,9.0,247.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BEAR CREEK CITY HALL-:-Marion,2072.0,1162.0,0.0,60.0,739.0,0.0,363.0,98.0,1051.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BRILLIANT COMM CTR-:-Marion,1185.0,694.0,0.0,34.0,490.0,0.0,170.0,46.0,640.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BYRD VOTING CTR-:-Marion,1359.0,836.0,0.0,57.0,507.0,0.0,272.0,80.0,744.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [53]:
comb_list = [i for i in comb.columns if ("REPRESENTATIVE" in i or "PRESIDENT" in i or "DELEGATES" in i or "STATEWIDE" in i or "CHIEF JUSTICE" in i or "APPEALS JUDGE" in i or "ASSOCIATE JUSTICE" in i) and ("Under Votes" not in i and "Over Votes" not in i)]

In [54]:
[i for i in comb.columns if i not in comb_list and "Over Votes" not in i and "Under Votes" not in i]

['index',
 'REGISTERED VOTERS - TOTAL-:-Registered Voters - Total',
 'BALLOTS CAST - TOTAL-:-Ballots Cast - Total',
 'BALLOTS CAST - BLANK-:-Ballots Cast - Blank',
 'STRAIGHT PARTY-:-Alabama Democratic Party-:-DEM',
 'STRAIGHT PARTY-:-Alabama Republican Party-:-REP',
 'STATE BOARD OF EDUCATION MEMBER DISTRICT 7-:-Allen Long-:-REP',
 'STATE BOARD OF EDUCATION MEMBER DISTRICT 7-:-Write-In-:-NON',
 'CIRCUIT COURT JUDGE, 25TH JUDICIAL CIRCUIT, PLACE 2-:-Talmage Lee Carter-:-REP',
 'CIRCUIT COURT JUDGE, 25TH JUDICIAL CIRCUIT, PLACE 2-:-Write-In-:-NON',
 'DISTRICT COURT JUDGE, MARION COUNTY-:-Mark Hammitte-:-REP',
 'DISTRICT COURT JUDGE, MARION COUNTY-:-Write-In-:-NON',
 'CIRCUIT CLERK, MARION COUNTY-:-Denise Ledbetter Mixon-:-REP',
 'CIRCUIT CLERK, MARION COUNTY-:-Write-In-:-NON',
 'MARION COUNTY JUDGE OF PROBATE-:-Paige Nichols Vick-:-REP',
 'MARION COUNTY JUDGE OF PROBATE-:-Write-In-:-NON',
 'SUPERINTENDENT, MARION COUNTY BOARD OF EDUCATION-:-Patrick Sutton-:-REP',
 'SUPERINTENDENT, MARIO

In [55]:
comb_list.sort()

In [56]:
comb_list

['ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Chris McCool-:-REP',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Write-In-:-NON',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Tommy Bryan-:-REP',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Write-In-:-NON',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Will Sellers-:-REP',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Write-In-:-NON',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Jay Mitchell-:-REP',
 'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Write-In-:-NON',
 'CHIEF JUSTICE OF THE SUPREME COURT-:-Greg Griffin-:-DEM',
 'CHIEF JUSTICE OF THE SUPREME COURT-:-Sarah Stewart-:-REP',
 'CHIEF JUSTICE OF THE SUPREME COURT-:-Write-In-:-NON',
 'COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Christy Edwards-:-REP',
 'COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Write-In-:-NON',
 'COURT OF CIVIL APPEALS JUDGE, PLACE 2-:-Chad Hanson-:-REP',
 'COURT OF CIVIL APPEALS JUDGE, PLACE 2-:-Write-In-:-NON',
 'COURT OF

In [57]:
remaining = [i for i in list(comb.columns) if i not in comb_list]

In [58]:
deleg_dict = {i:"PRESIDENT OF THE UNITED STATES-:-"+i.split("-:-")[1]+"-:-"+i.split("-:-")[2] for i in comb_list if "DEMOCRATIC DELEGATES" in i}

In [59]:
comb = comb[["index"]+comb_list]

In [60]:
comb.rename(columns = deleg_dict, inplace = True)

In [61]:
comb2 = comb.groupby(level = 0,axis = 1).sum()

In [62]:
comb2.columns

Index(['ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Chris McCool-:-REP',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Write-In-:-NON',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Tommy Bryan-:-REP',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Write-In-:-NON',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Will Sellers-:-REP',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Write-In-:-NON',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Jay Mitchell-:-REP',
       'ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Write-In-:-NON',
       'CHIEF JUSTICE OF THE SUPREME COURT-:-Greg Griffin-:-DEM',
       'CHIEF JUSTICE OF THE SUPREME COURT-:-Sarah Stewart-:-REP',
       'CHIEF JUSTICE OF THE SUPREME COURT-:-Write-In-:-NON',
       'COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Christy Edwards-:-REP',
       'COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Write-In-:-NON',
       'COURT OF CIVIL APPEALS JUDGE, PLACE 2-:-Chad 

In [63]:
def get_race(race_string):
    race_string = race_string.title()
    race_string = race_string.replace("(Vote For 1)","")
    if "U.S. House" in race_string or 'Us House' in race_string or "United States Representative" in race_string:
        return "CON"
    elif "Public Service" in race_string:
        if "1" in race_string:
            return "PS1"
        elif "2" in race_string:
            return "PS2"
        else:
            return "PSC"
    elif "United States Senator" in race_string:
        return "USS"
    elif "State House" in race_string or "State Representative" in race_string:
        return "SL"
    elif "State Senator" in race_string:
        return "SU"
    elif "President" in race_string:
        return "PRE"
    elif "US Senate" in race_string or "Us Senate" in race_string:
        return "USS"

    elif "Attorney General" in race_string:
        return "ATG"
    elif "Auditor General" in race_string or "State Auditor" in race_string:
        return "AUD"
    elif "Treasurer" in race_string:
        return "TRE"
    elif "Superintendent" in race_string:
        return "SUP"
    elif "Secretary Of State" in race_string:
        return "SOS"
    elif "Lieutenant Governor" in race_string:
        return "LTG"
    elif "Governor" in race_string:
        return "GOV"
    elif "Commissioner Of Labor" in race_string:
        return "LAB"
    elif "Commissioner Of Agriculture" in race_string:
        return "AGR"
    elif "Commissioner Of Insurance" in race_string:
        return "INS"
    elif "Associate Justice Of The Supreme Court" in race_string:
        if "1" in race_string:
            return "AJ1"
        elif "2" in race_string:
            return "AJ2"
        elif "3" in race_string:
            return "AJ3"
        elif "4" in race_string:
            return "AJ4"
    elif "Constitution" in race_string:
        return "CNS"
    elif "Chief Justice" in race_string:
        return "CFJ"
    elif "Amendment" in race_string:
        if "One " in race_string:
            return "A01"
        elif "Two " in race_string:
            return "A02"
        elif "Three " in race_string:
            return "A03"
        elif "Four " in race_string:
            return "A04"
        elif "Five " in race_string:
            return "A05"
        elif "Six " in race_string:
            return "A06"
        elif "Seven " in race_string:
            return "A07"
        elif "Eight " in race_string:
            return "A08"
        elif "Nine " in race_string:
            return "A09"
        elif "Ten " in race_string:
            return "A10"
        else:
            print("No race for:", race_string)
            raise ValueError
    elif "Referendum" in race_string:
        if "1" in race_string or "A" in race_string:
            return "RFA"
        elif "2" in race_string or "B" in race_string:
            return "RFB"
        else:
            print("No race for:", race_string)
            raise ValueError
    elif "Civil Appeals Judge" in race_string:
        if "1" in race_string:
            return "CV1"
        elif "2" in race_string:
            return "CV2"
        elif "3" in race_string:
            return "CV3"
        else:
            raise ValueError
    elif "Criminal Appeals Judge" in race_string:
        if "1" in race_string:
            return "CR1"
        elif "2" in race_string:
            return "CR2"
        elif "3" in race_string:
            return "CR3"
        else:
            raise ValueError
    else:
        print("No race for:", race_string)
        raise ValueError
        
def get_election_type(race_string):
    if "(runoff)" in race_string:
        return "R"
    else:
        return "G"
        
def get_party(race_string):
    if "REP" in race_string:
        return "R"
    elif "DEM" in race_string or "(Democrat)" in race_string:
        return "D"
    elif "LIB" in race_string or "(L)" in race_string:
        return "L"
    elif "NON" in race_string:
        return "O"
    elif "IND" in race_string:
        return "I"
    elif race_string[0:3]=="Yes":
        return "YES"
    elif race_string[0:2]=="No":
        return "NO"
    else:
        print("NO RACE", race_string)
        return ""
           
def get_name(name_string):
    if "Write-In" in name_string:
        return "WRI"
    if "AMENDMENT" not in name_string and "Referendum" not in name_string and "CONSTITUTION" not in name_string:
        #print(name_string)
        #name_string = name_string.split(" (")[0]
        name_string = name_string.replace("'","")
        likely_last = name_string.split(" ")[-1]
        proposed_last = likely_last[:3]
        if proposed_last in ['II', 'III', 'Jr', 'Jr.', 'Sr.', 'JR.', "JR", "IV","Jr-"]:
            likely_last = name_string.split(" ")[-2]
            proposed_last = likely_last[:3]
        #print(proposed_last.upper())
        return proposed_last.upper()
    else:
        return name_string.split("-:-")[1].upper()
#     name_string = name_string.split("-:-")[1]
#     name_string = name_string.replace(" (I)","")
#     name_string = name_string.replace("'","")
#     likely_last = name_string.split(" ")[-1]
#     proposed_last = likely_last[:3]
#     if proposed_last in ['II', 'III', 'Jr', 'Jr.', 'Sr.', 'JR.', "JR", "IV"]:
#         likely_last = name_string.split(" ")[-2]
#         proposed_last = likely_last[:3]
#     return proposed_last.upper()

def get_district(race_string, fill_level):
    race_string = race_string.split("-:-")[0]
    race_string = race_string.replace(" (Vote For 1)","")
    if "UNITED STATES REPRESENTATIVE" in race_string:
        break_word = "REPRESENTATIVE, "
        temp = race_string.split(break_word)[1]
        temp = re.findall('\d*', temp)[0]
    elif "STATE REPRESENTATIVE" in race_string or "STATE SENATOR" in race_string:
        break_word = "DISTRICT "
        temp = race_string.split(break_word)[1]
    else:
        raise ValueError
    
    return temp.zfill(fill_level)

def column_rename_function(name_string):
    election_type = "G"
    year = "24"
    party = get_party(name_string.split("-:-")[-1])
    race = get_race(name_string)
    district = ""
    if race in ["CON", "SU"]:
        district = get_district(name_string, 2)
        year = ""
    elif race in ["SL"]:
        district = get_district(name_string, 3)
        year = ""
    
    name = get_name(name_string)
    if "CONSTITUTION" in name_string or "STATEWIDE AMENDMENT" in name_string:

        new_col_name = election_type + year + race + district +  name
    else:
        new_col_name = election_type + year + race + district + party + name
        print(election_type)
        print(year)
        print(race)
        print(district)
        print(name)
    if len(new_col_name) > 10:
        print(name_string)
        print(new_col_name)
    return new_col_name

# Make a dictionary that points to the new column names and checks for duplicates
race_columns = [i for i in list(comb2.columns) if i != 'index']

race_updates_dict = {}
race_updates_reversed = {}
clean_dups = {}
new_names = []
for val in race_columns:
    print(val)
    new_name = column_rename_function(val)
    race_updates_dict[val] = new_name
    if new_name not in new_names:
        new_names.append(new_name)
        race_updates_reversed[new_name] = val
    else:
        print("Duplicate", new_name)
        print(race_updates_reversed[new_name])
        print(val)
        clean_dups[val] = race_updates_reversed[new_name]

ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Chris McCool-:-REP
G
24
AJ1

MCC
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Write-In-:-NON
G
24
AJ1

WRI
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Tommy Bryan-:-REP
G
24
AJ2

BRY
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Write-In-:-NON
G
24
AJ2

WRI
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Will Sellers-:-REP
G
24
AJ3

SEL
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Write-In-:-NON
G
24
AJ3

WRI
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Jay Mitchell-:-REP
G
24
AJ4

MIT
ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Write-In-:-NON
G
24
AJ4

WRI
CHIEF JUSTICE OF THE SUPREME COURT-:-Greg Griffin-:-DEM
G
24
CFJ

GRI
CHIEF JUSTICE OF THE SUPREME COURT-:-Sarah Stewart-:-REP
G
24
CFJ

STE
CHIEF JUSTICE OF THE SUPREME COURT-:-Write-In-:-NON
G
24
CFJ

WRI
COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Christy Edwards-:-REP
G
24
CV1

EDW
COURT OF CIVIL APPEALS JUDGE, PLACE 1-:-Write-In-:-NON
G
24
CV1

WRI
C

In [64]:
myKeys = list(race_updates_dict.values())
myKeys.sort()
sorted_dict = dict(sorted(race_updates_dict.items(), key=lambda x:x[1]))
export_dict = {i:key for key, i in sorted_dict.items()}

In [65]:
pd.DataFrame(export_dict.items()).to_csv("./field_names_gen.csv", index = False)


In [66]:
comb = comb2.copy(deep = True)

In [67]:
comb

pivot_col,"ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Chris McCool-:-REP","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 1-:-Write-In-:-NON","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Tommy Bryan-:-REP","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 2-:-Write-In-:-NON","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Will Sellers-:-REP","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 3-:-Write-In-:-NON","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Jay Mitchell-:-REP","ASSOCIATE JUSTICE OF THE SUPREME COURT, PLACE 4-:-Write-In-:-NON",CHIEF JUSTICE OF THE SUPREME COURT-:-Greg Griffin-:-DEM,CHIEF JUSTICE OF THE SUPREME COURT-:-Sarah Stewart-:-REP,...,"UNITED STATES REPRESENTATIVE, 4TH CONGRESSIONAL DISTRICT-:-Write-In-:-NON","UNITED STATES REPRESENTATIVE, 5TH CONGRESSIONAL DISTRICT-:-Dale Strong-:-REP","UNITED STATES REPRESENTATIVE, 5TH CONGRESSIONAL DISTRICT-:-Write-In-:-NON","UNITED STATES REPRESENTATIVE, 6TH CONGRESSIONAL DISTRICT-:-Elizabeth Anderson-:-DEM","UNITED STATES REPRESENTATIVE, 6TH CONGRESSIONAL DISTRICT-:-Gary Palmer-:-REP","UNITED STATES REPRESENTATIVE, 6TH CONGRESSIONAL DISTRICT-:-Write-In-:-NON","UNITED STATES REPRESENTATIVE, 7TH CONGRESSIONAL DISTRICT-:-Robin Litaker-:-REP","UNITED STATES REPRESENTATIVE, 7TH CONGRESSIONAL DISTRICT-:-Terri A. Sewell-:-DEM","UNITED STATES REPRESENTATIVE, 7TH CONGRESSIONAL DISTRICT-:-Write-In-:-NON",index
0,390.0,3.0,389.0,4.0,389.0,3.0,389.0,4.0,55.0,384.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ABSENTEE-:-Marion
1,241.0,0.0,241.0,0.0,241.0,0.0,240.0,1.0,10.0,240.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BACCUS UNION HALL-:-Marion
2,1047.0,8.0,1045.0,9.0,1046.0,8.0,1046.0,8.0,110.0,1021.0,...,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BEAR CREEK CITY HALL-:-Marion
3,629.0,1.0,629.0,1.0,628.0,0.0,627.0,0.0,51.0,628.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BRILLIANT COMM CTR-:-Marion
4,731.0,2.0,727.0,1.0,723.0,2.0,722.0,2.0,87.0,709.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BYRD VOTING CTR-:-Marion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,142.0,0.0,142.0,0.0,142.0,0.0,141.0,0.0,21.0,137.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SRYGLEY CH OF CHRIST-:-Colbert
32,614.0,16.0,611.0,14.0,611.0,14.0,610.0,13.0,217.0,574.0,...,19.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TREE HOUSE SCH-:-Colbert
33,671.0,15.0,671.0,13.0,673.0,12.0,667.0,13.0,315.0,615.0,...,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TUSCUMBIA MULTI-PURPOSE-:-Colbert
34,537.0,7.0,536.0,8.0,536.0,6.0,537.0,4.0,115.0,518.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,WESTSIDE BAPT_-:-Colbert


In [68]:
myKeys = list(race_updates_dict.values())
myKeys.sort()
#sorted_dict = {myKeys: race_updates_dict[i] for i in myKeys}


sorted_dict = dict(sorted(race_updates_dict.items(), key=lambda x:x[1]))
comb.rename(columns = race_updates_dict, inplace = True)


In [69]:
comb

pivot_col,G24AJ1RMCC,G24AJ1OWRI,G24AJ2RBRY,G24AJ2OWRI,G24AJ3RSEL,G24AJ3OWRI,G24AJ4RMIT,G24AJ4OWRI,G24CFJDGRI,G24CFJRSTE,...,GCON04OWRI,GCON05RSTR,GCON05OWRI,GCON06DAND,GCON06RPAL,GCON06OWRI,GCON07RLIT,GCON07DSEW,GCON07OWRI,index
0,390.0,3.0,389.0,4.0,389.0,3.0,389.0,4.0,55.0,384.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ABSENTEE-:-Marion
1,241.0,0.0,241.0,0.0,241.0,0.0,240.0,1.0,10.0,240.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BACCUS UNION HALL-:-Marion
2,1047.0,8.0,1045.0,9.0,1046.0,8.0,1046.0,8.0,110.0,1021.0,...,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BEAR CREEK CITY HALL-:-Marion
3,629.0,1.0,629.0,1.0,628.0,0.0,627.0,0.0,51.0,628.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BRILLIANT COMM CTR-:-Marion
4,731.0,2.0,727.0,1.0,723.0,2.0,722.0,2.0,87.0,709.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BYRD VOTING CTR-:-Marion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,142.0,0.0,142.0,0.0,142.0,0.0,141.0,0.0,21.0,137.0,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SRYGLEY CH OF CHRIST-:-Colbert
32,614.0,16.0,611.0,14.0,611.0,14.0,610.0,13.0,217.0,574.0,...,19.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TREE HOUSE SCH-:-Colbert
33,671.0,15.0,671.0,13.0,673.0,12.0,667.0,13.0,315.0,615.0,...,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TUSCUMBIA MULTI-PURPOSE-:-Colbert
34,537.0,7.0,536.0,8.0,536.0,6.0,537.0,4.0,115.0,518.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,WESTSIDE BAPT_-:-Colbert


In [70]:
comb = comb[["index"] + myKeys]

In [71]:
comb

pivot_col,index,G24A01NO,G24A01YES,G24AJ1OWRI,G24AJ1RMCC,G24AJ2OWRI,G24AJ2RBRY,G24AJ3OWRI,G24AJ3RSEL,G24AJ4OWRI,...,GCON04OWRI,GCON04RADE,GCON05OWRI,GCON05RSTR,GCON06DAND,GCON06OWRI,GCON06RPAL,GCON07DSEW,GCON07OWRI,GCON07RLIT
0,ABSENTEE-:-Marion,55.0,202.0,3.0,390.0,4.0,389.0,3.0,389.0,4.0,...,2.0,399.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BACCUS UNION HALL-:-Marion,79.0,112.0,0.0,241.0,0.0,241.0,0.0,241.0,1.0,...,0.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BEAR CREEK CITY HALL-:-Marion,143.0,757.0,8.0,1047.0,9.0,1045.0,8.0,1046.0,8.0,...,15.0,1056.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BRILLIANT COMM CTR-:-Marion,125.0,387.0,1.0,629.0,1.0,629.0,0.0,628.0,0.0,...,2.0,636.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BYRD VOTING CTR-:-Marion,145.0,437.0,2.0,731.0,1.0,727.0,2.0,723.0,2.0,...,5.0,744.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31,SRYGLEY CH OF CHRIST-:-Colbert,21.0,103.0,0.0,142.0,0.0,142.0,0.0,142.0,0.0,...,5.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,TREE HOUSE SCH-:-Colbert,89.0,484.0,16.0,614.0,14.0,611.0,14.0,611.0,13.0,...,19.0,626.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
33,TUSCUMBIA MULTI-PURPOSE-:-Colbert,97.0,629.0,15.0,671.0,13.0,671.0,12.0,673.0,13.0,...,17.0,687.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
34,WESTSIDE BAPT_-:-Colbert,72.0,429.0,7.0,537.0,8.0,536.0,6.0,536.0,4.0,...,10.0,541.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
export_dict = {i:key for key, i in sorted_dict.items()}

In [73]:
pd.DataFrame(export_dict.items()).to_csv("./field_names.csv", index = False)

In [74]:
comb["County"] = comb["index"].apply(lambda x:x.split("-:-")[1])

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/3688505148.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["County"] = comb["index"].apply(lambda x:x.split("-:-")[1])


In [75]:
comb.groupby("County").sum().to_csv("./al_county_totals.csv")

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/1310031185.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  comb.groupby("County").sum().to_csv("./al_county_totals.csv")


In [76]:
for i in list(sorted_dict.values()):
    comb[i] = comb[i].astype(int)
    print(i, sum(comb[i]))
    

G24A01NO 399640
G24A01YES 1159794
G24AJ1OWRI 38088
G24AJ1RMCC 1564832
G24AJ2OWRI 36383
G24AJ2RBRY 1560072
G24AJ3OWRI 35479
G24AJ3RSEL 1557606
G24AJ4OWRI 35034
G24AJ4RMIT 1555235
G24CFJDGRI 756675
G24CFJOWRI 2350
G24CFJRSTE 1458501
G24CR1OWRI 33207
G24CR1RMIN 1547401
G24CR2OWRI 32940
G24CR2RAND 1546549
G24CR3OWRI 32746
G24CR3RCOL 1547520
G24CV1OWRI 33716
G24CV1REDW 1555361
G24CV2OWRI 33644
G24CV2RHAN 1549864
G24CV3OWRI 33096
G24CV3RMOO 1551657
G24PREDHAR 772412
G24PREIKEN 12075
G24PREIOLI 4930
G24PREISTE 4319
G24PREOWRI 8738
G24PRERTRU 1462616
G24PSCOWRI 42061
G24PSCRCAV 1538888
GCON01DHOL 70929
GCON01OWRI 306
GCON01RMOO 258619
GCON02DFIG 158041
GCON02OWRI 219
GCON02RDOB 131414
GCON03OWRI 5160
GCON03RROG 243848
GCON04OWRI 3374
GCON04RADE 274498
GCON05OWRI 12088
GCON05RSTR 250322
GCON06DAND 102504
GCON06OWRI 380
GCON06RPAL 243741
GCON07DSEW 186723
GCON07OWRI 185
GCON07RLIT 106312


/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/2293604008.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb[i] = comb[i].astype(int)


In [77]:
county = pd.read_csv("/Users/peterhorton/Documents/RDH/raw_data/census/PL_COUNTYFP_NAMES.csv")
al_cnty_fips_dict = dict((zip(county[county["STUSAB"]=="AL"]["NAME"],county[county["STUSAB"]=="AL"]["COUNTYFP"].astype(str).str.zfill(3))))

In [78]:
comb["Precinct"] = comb["index"].apply(lambda x: x.split("-:-")[0])

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/545608281.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["Precinct"] = comb["index"].apply(lambda x: x.split("-:-")[0])


In [79]:
comb["County"] = comb["index"].apply(lambda x: x.split("-:-")[1])

comb["County_map"] = comb["County"] + " County"

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/2491721802.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["County"] = comb["index"].apply(lambda x: x.split("-:-")[1])
/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/2491721802.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["County_map"] = comb["County"] + " County"


In [80]:
comb["COUNTYFP"] = comb["County_map"].map(al_cnty_fips_dict).fillna(comb["County"])

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/3302747155.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["COUNTYFP"] = comb["County_map"].map(al_cnty_fips_dict).fillna(comb["County"])


In [81]:
comb["index"] = comb["index"].apply(lambda x: x.split("-:-")[1]+"-:-"+x.split("-:-")[0])

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/3319974299.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["index"] = comb["index"].apply(lambda x: x.split("-:-")[1]+"-:-"+x.split("-:-")[0])


In [82]:
comb.rename(columns = {"index":"UNIQUE_ID"}, inplace = True)

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/3538977619.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb.rename(columns = {"index":"UNIQUE_ID"}, inplace = True)


In [83]:
comb = comb[["UNIQUE_ID","COUNTYFP","County","Precinct"]+list(sorted_dict.values())]

In [84]:
comb["COUNTYFP"].unique()

array(['093', '063', '071', '091', '053', '077', '015', '099', '127',
       '049', '007', '043', '087', '131', '017', '111', '009', '045',
       '105', '039', '065', '085', '013', '123', '037', '107', '019',
       '081', '031', '029', '041', '069', '001', '119', '129', '083',
       '025', '047', '075', '073', '113', '003', '005', '125', '079',
       '057', '103', '115', '061', '101', '097', '023', '027', '117',
       '055', '035', '051', '089', '059', '095', '011', '121', '133',
       '021', '109', '067', '033'], dtype=object)

In [85]:
al_cnty_fips_dict

{'Autauga County': '001',
 'Baldwin County': '003',
 'Barbour County': '005',
 'Bibb County': '007',
 'Blount County': '009',
 'Bullock County': '011',
 'Butler County': '013',
 'Calhoun County': '015',
 'Chambers County': '017',
 'Cherokee County': '019',
 'Chilton County': '021',
 'Choctaw County': '023',
 'Clarke County': '025',
 'Clay County': '027',
 'Cleburne County': '029',
 'Coffee County': '031',
 'Colbert County': '033',
 'Conecuh County': '035',
 'Coosa County': '037',
 'Covington County': '039',
 'Crenshaw County': '041',
 'Cullman County': '043',
 'Dale County': '045',
 'Dallas County': '047',
 'DeKalb County': '049',
 'Elmore County': '051',
 'Escambia County': '053',
 'Etowah County': '055',
 'Fayette County': '057',
 'Franklin County': '059',
 'Geneva County': '061',
 'Greene County': '063',
 'Hale County': '065',
 'Henry County': '067',
 'Houston County': '069',
 'Jackson County': '071',
 'Jefferson County': '073',
 'Lamar County': '075',
 'Lauderdale County': '077',
 

In [86]:
comb["COUNTYFP"].unique()

array(['093', '063', '071', '091', '053', '077', '015', '099', '127',
       '049', '007', '043', '087', '131', '017', '111', '009', '045',
       '105', '039', '065', '085', '013', '123', '037', '107', '019',
       '081', '031', '029', '041', '069', '001', '119', '129', '083',
       '025', '047', '075', '073', '113', '003', '005', '125', '079',
       '057', '103', '115', '061', '101', '097', '023', '027', '117',
       '055', '035', '051', '089', '059', '095', '011', '121', '133',
       '021', '109', '067', '033'], dtype=object)

In [87]:
comb["UNIQUE_ID"] = comb["COUNTYFP"] + "-:-" + comb["Precinct"]

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/303240413.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb["UNIQUE_ID"] = comb["COUNTYFP"] + "-:-" + comb["Precinct"]


In [88]:
comb.sort_values(["COUNTYFP","Precinct"], inplace = True)

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/2103180171.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comb.sort_values(["COUNTYFP","Precinct"], inplace = True)


In [89]:
comb.groupby("County").sum().to_csv("./prec_county_totals_gen.csv", index = True)

/var/folders/1t/0q4w6hm92mg_zxd84dfxmq3m0000gn/T/ipykernel_6180/2354233316.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  comb.groupby("County").sum().to_csv("./prec_county_totals_gen.csv", index = True)


In [90]:
import os

if not os.path.exists("./al_2024_gen_prec_csv"):
    os.mkdir("./al_2024_gen_prec_csv")

comb.to_csv("./al_2024_gen_prec_csv/al_2024_gen_prec_csv.csv", index = False)

In [ ]:
comb

In [91]:
import shutil

shutil.make_archive("./al_2024_gen_prec_csv/", "zip", "./al_2024_gen_prec_csv/")

'/Users/peterhorton/Documents/RDH/pber_local/AL_2024/general/al_2024_gen_prec_csv.zip'